In [1]:
import sys
print(sys.executable)

/Users/jairamdulasi/llm-zoomcamp-2026/module-06/.venv/bin/python


In [15]:
from dotenv import load_dotenv
load_dotenv()

import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: https://logfire-us.pydantic.dev/jairamd/llm-zoomcamp-2026


In [16]:
from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data

documents = load_faq_data()
index = build_index(documents)
deps = SearchDeps(index=index)

query = "How do I run Ollama locally?"
response = await faq_agent.run(query, deps=deps)
print(response.output)

18:41:53.634 faq_agent run
18:41:53.640   chat gpt-5.4-mini
18:41:55.359   running tool: search
18:41:55.372   chat gpt-5.4-mini
Yes — you can run Ollama locally.

From the course FAQ:

- **Install Ollama** from: https://ollama.com/download
  - **macOS**: download and install the `.pkg`
  - **Windows**: download and install the `.msi`
  - **Linux**: run:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

- Then start a model locally with:
  ```bash
  ollama run llama3
  ```

  This downloads the model and opens a local chat interface.

- To test that the local Ollama server is running:
  ```bash
  curl http://localhost:11434
  ```

  You should see a response like:
  ```json
  {"models": [...]}
  ```

- If you want to use it from Python:
  ```bash
  pip install ollama
  ```

  Example:
  ```python
  import ollama

  response = ollama.chat(
      model='llama3',
      messages=[{"role": "user", "content": your_prompt}]
  )

  print(response['message']['content'])
  `

## Q1
Checked the trace in the Logfire UI (project: llm-zoomcamp-2026): the trace for this run contains
4 spans — `faq_agent run` (root), `chat gpt-5.4-mini`, `running tool: search`, `chat gpt-5.4-mini`.

**Answer: 5**

In [17]:
import duckdb

con = duckdb.connect("logfire_pipeline.duckdb")

table_count = con.execute("""
    SELECT COUNT(*) FROM information_schema.tables 
    WHERE table_schema = 'agent_traces'
""").fetchone()
print(table_count)

(24,)


In [18]:
trace_id = "019f8756e0b3fc6c29e58bbfe0402686"

result = con.execute(f"""
    SELECT SUM(attributes__gen_ai_usage_input_tokens)
    FROM agent_traces.records
    WHERE trace_id = '{trace_id}'
""").fetchone()
print(result)

(1827,)
